# 03 基线建模（Phase 3）

这个 notebook 对应 `Agent.md` 里的 **Phase 3: Baseline Modeling**。当前版本的 baseline 不再只训练一个 logistic regression，而是固定在 **同一模型家族** 下比较两套数据口径：

- `traditional_core`
- `traditional_plus_proxy`

因此 Phase 3 的结论必须写成 **数据增量对比**，不能把 uplift 误解为模型家族提升。


## 1. 运行参数与辅助函数

这里默认沿用 Phase 2 的两套 preprocessing artifact，并在同一 validation 集上训练两套 logistic baseline。


In [ ]:
from pathlib import Path
import json
import sys
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse

root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "pyproject.toml").exists()),
    Path.cwd(),
)
src = root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

warnings.filterwarnings(
    "ignore",
    message="Pandas requires version '1.3.6' or newer of 'bottleneck'",
)

from credit_visable.config import load_settings
from credit_visable.features import FEATURE_SET_NAMES as SUPPORTED_FEATURE_SET_NAMES
from credit_visable.modeling import (
    build_binary_diagnostic_curves,
    evaluate_binary_classifier,
    train_logistic_baseline,
)
from credit_visable.utils.paths import get_paths

settings = load_settings()
paths = get_paths()

FEATURE_SET_NAMES = list(SUPPORTED_FEATURE_SET_NAMES)
PREPROCESSING_OUTPUT_ROOT = paths.data_processed / "preprocessing"
MODELING_OUTPUT_ROOT = paths.data_processed / "modeling_baseline"
FIGURE_OUTPUT_DIR = paths.reports_figures

OPERATING_THRESHOLD = 0.50
SCENARIO_THRESHOLDS = [0.30, 0.50, 0.70]
MODEL_KWARGS = {}
RANDOM_STATE = settings.random_state

PHASE_3_READY = False
phase3_load_ok = False
figure_paths = {}
artifacts_by_set = {}
validation_scores_by_set = {}
metrics_payload_by_set = {}
comparison_metrics = pd.DataFrame()
data_uplift_summary = pd.DataFrame()
threshold_scenarios = pd.DataFrame()
operating_threshold_comparison = pd.DataFrame()
validation_score_comparison = pd.DataFrame()


def build_required_files(output_dir: Path) -> dict[str, Path]:
    return {
        "X_train": output_dir / "X_train.npz",
        "X_valid": output_dir / "X_valid.npz",
        "train_meta": output_dir / "train_meta.csv",
        "valid_meta": output_dir / "valid_meta.csv",
        "feature_names": output_dir / "feature_names.csv",
        "manifest": output_dir / "manifest.json",
        "feature_set_manifest": output_dir / "feature_set_manifest.json",
    }


def _safe_mean(values: np.ndarray) -> float | None:
    if values.size == 0:
        return None
    return float(np.mean(values))


def _matrix_density(matrix: sparse.spmatrix) -> float:
    total_cells = matrix.shape[0] * matrix.shape[1]
    if total_cells == 0:
        return 0.0
    return float(matrix.nnz / total_cells)


def _to_builtin(value):
    if isinstance(value, dict):
        return {str(key): _to_builtin(item) for key, item in value.items()}
    if isinstance(value, list):
        return [_to_builtin(item) for item in value]
    if isinstance(value, tuple):
        return [_to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return [_to_builtin(item) for item in value.tolist()]
    if isinstance(value, (np.floating, float)):
        if not np.isfinite(value):
            return None
        return float(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if pd.isna(value):
        return None
    return value


def build_threshold_scenarios(
    y_true: np.ndarray,
    y_score: np.ndarray,
    feature_set_name: str,
) -> pd.DataFrame:
    y_true_array = np.asarray(y_true)
    y_score_array = np.asarray(y_score)
    total_bad = int(y_true_array.sum())
    rows = []

    for threshold in SCENARIO_THRESHOLDS:
        scenario_metrics = evaluate_binary_classifier(
            y_true_array,
            y_score_array,
            threshold=threshold,
        )
        approved_mask = y_score_array < threshold
        rejected_mask = ~approved_mask
        rows.append(
            {
                "feature_set": feature_set_name,
                "model_family": "logistic",
                "model_label": f"logistic_{feature_set_name}",
                "threshold": float(threshold),
                "approval_rate": float(approved_mask.mean()),
                "reject_rate": float(rejected_mask.mean()),
                "approved_population_bad_rate": _safe_mean(y_true_array[approved_mask]),
                "rejected_population_bad_capture_rate": (
                    float(y_true_array[rejected_mask].sum() / total_bad)
                    if total_bad > 0
                    else 0.0
                ),
                "precision": scenario_metrics["precision"],
                "recall": scenario_metrics["recall"],
                "f1": scenario_metrics["f1"],
            }
        )

    return pd.DataFrame(rows)


print(f"Preprocessing output root: {PREPROCESSING_OUTPUT_ROOT}")
print(f"Modeling output root: {MODELING_OUTPUT_ROOT}")
print(f"Figure output dir: {FIGURE_OUTPUT_DIR}")
print(f"Feature sets: {FEATURE_SET_NAMES}")


## 2. Readiness gate

Phase 3 不再接受单臂输入。只有当两套 Phase 2 artifact 都完整时，baseline comparison 才能继续。


In [ ]:
required_files_by_set = {
    feature_set_name: build_required_files(PREPROCESSING_OUTPUT_ROOT / feature_set_name)
    for feature_set_name in FEATURE_SET_NAMES
}
readiness_rows = []
for feature_set_name, required_files in required_files_by_set.items():
    readiness_rows.extend(
        [
            {
                "feature_set": feature_set_name,
                "artifact": artifact_name,
                "path": str(path),
                "exists": path.exists(),
            }
            for artifact_name, path in required_files.items()
        ]
    )

readiness_frame = pd.DataFrame(readiness_rows)
PHASE_3_READY = bool(readiness_frame["exists"].all())
display(readiness_frame)
print(f"Phase 3 ready: {PHASE_3_READY}")

if not PHASE_3_READY:
    missing_files = readiness_frame.loc[~readiness_frame["exists"]]
    display(missing_files)
    print("Phase 3 blocked. 请先完成 notebooks/02_preprocessing.ipynb，并确认两套 feature set 的标准产物都已生成。")


## 3. 加载并校验两套 preprocessing artifact

这里除了做单套 artifact 的完整性检查，还要验证两套 feature set 的 train / valid 主键和目标完全对齐，否则 uplift 对比没有解释价值。


In [ ]:
if not PHASE_3_READY:
    print("加载步骤已跳过，因为 Phase 2 产物不完整。")
else:
    load_summary_rows = []
    validation_errors = []
    reference_train_meta = None
    reference_valid_meta = None

    for feature_set_name, required_files in required_files_by_set.items():
        X_train = sparse.load_npz(required_files["X_train"])
        X_valid = sparse.load_npz(required_files["X_valid"])
        train_meta = pd.read_csv(required_files["train_meta"])
        valid_meta = pd.read_csv(required_files["valid_meta"])
        feature_names = pd.read_csv(required_files["feature_names"])
        manifest = json.loads(required_files["manifest"].read_text(encoding="utf-8"))
        feature_set_manifest = json.loads(
            required_files["feature_set_manifest"].read_text(encoding="utf-8")
        )

        feature_name_column = (
            "feature_name" if "feature_name" in feature_names.columns else feature_names.columns[0]
        )
        feature_name_list = feature_names[feature_name_column].astype(str).tolist()

        if settings.target_column not in train_meta.columns:
            validation_errors.append(f"{feature_set_name}: train_meta 缺少目标列")
        if settings.target_column not in valid_meta.columns:
            validation_errors.append(f"{feature_set_name}: valid_meta 缺少目标列")
        if settings.id_column not in train_meta.columns:
            validation_errors.append(f"{feature_set_name}: train_meta 缺少主键列")
        if settings.id_column not in valid_meta.columns:
            validation_errors.append(f"{feature_set_name}: valid_meta 缺少主键列")
        if X_train.shape[0] != len(train_meta):
            validation_errors.append(f"{feature_set_name}: X_train 行数与 train_meta 行数不一致")
        if X_valid.shape[0] != len(valid_meta):
            validation_errors.append(f"{feature_set_name}: X_valid 行数与 valid_meta 行数不一致")
        if X_train.shape[1] == 0 or X_valid.shape[1] == 0:
            validation_errors.append(f"{feature_set_name}: 没有可训练特征列")
        if X_train.shape[1] != X_valid.shape[1]:
            validation_errors.append(f"{feature_set_name}: 训练集和验证集特征维度不一致")
        if len(feature_name_list) != X_train.shape[1]:
            validation_errors.append(f"{feature_set_name}: feature_names 条数与矩阵列数不一致")
        if manifest.get("train_shape") != list(X_train.shape):
            validation_errors.append(f"{feature_set_name}: manifest 中 train_shape 与 X_train 不一致")
        if manifest.get("valid_shape") != list(X_valid.shape):
            validation_errors.append(f"{feature_set_name}: manifest 中 valid_shape 与 X_valid 不一致")
        if feature_set_manifest.get("feature_set_name") != feature_set_name:
            validation_errors.append(f"{feature_set_name}: feature_set_manifest 的 feature_set_name 不一致")

        sorted_train_meta = train_meta[[settings.id_column, settings.target_column]].sort_values(settings.id_column).reset_index(drop=True)
        sorted_valid_meta = valid_meta[[settings.id_column, settings.target_column]].sort_values(settings.id_column).reset_index(drop=True)
        if reference_train_meta is None:
            reference_train_meta = sorted_train_meta
            reference_valid_meta = sorted_valid_meta
        else:
            if not reference_train_meta.equals(sorted_train_meta):
                validation_errors.append(f"{feature_set_name}: train_meta 与参考 feature set 的主键/目标不一致")
            if not reference_valid_meta.equals(sorted_valid_meta):
                validation_errors.append(f"{feature_set_name}: valid_meta 与参考 feature set 的主键/目标不一致")

        artifacts_by_set[feature_set_name] = {
            "X_train": X_train,
            "X_valid": X_valid,
            "train_meta": train_meta,
            "valid_meta": valid_meta,
            "feature_names": feature_name_list,
            "manifest": manifest,
            "feature_set_manifest": feature_set_manifest,
        }
        load_summary_rows.extend(
            [
                {
                    "feature_set": feature_set_name,
                    "artifact": "X_train",
                    "shape": X_train.shape,
                    "density": _matrix_density(X_train),
                },
                {
                    "feature_set": feature_set_name,
                    "artifact": "X_valid",
                    "shape": X_valid.shape,
                    "density": _matrix_density(X_valid),
                },
                {
                    "feature_set": feature_set_name,
                    "artifact": "train_meta",
                    "shape": train_meta.shape,
                    "density": None,
                },
                {
                    "feature_set": feature_set_name,
                    "artifact": "valid_meta",
                    "shape": valid_meta.shape,
                    "density": None,
                },
            ]
        )

    phase3_load_ok = not validation_errors
    display(pd.DataFrame(load_summary_rows))
    if validation_errors:
        display(pd.DataFrame({"validation_error": validation_errors}))
        print("Phase 3 stopped because the saved Phase 2 artifacts failed validation.")
    else:
        print("Phase 3 load validation passed for both feature sets.")


## 4. 训练两套 logistic baseline，并做数据增量对比

这里固定模型家族为 logistic regression。所有 uplift 都应解释为 **`traditional_plus_proxy` 相对 `traditional_core` 的数据增量**。


In [ ]:
if not PHASE_3_READY:
    print("训练与评分已跳过，因为 Phase 2 产物不完整。")
elif not phase3_load_ok:
    print("训练与评分已跳过，因为加载校验未通过。")
else:
    comparison_rows = []
    threshold_frames = []

    for feature_set_name in FEATURE_SET_NAMES:
        bundle = artifacts_by_set[feature_set_name]
        y_train = bundle["train_meta"][settings.target_column]
        y_valid = bundle["valid_meta"][settings.target_column].to_numpy()

        model = train_logistic_baseline(
            bundle["X_train"],
            y_train,
            random_state=RANDOM_STATE,
            **MODEL_KWARGS,
        )
        y_score = model.predict_proba(bundle["X_valid"])[:, 1]

        primary_metrics = evaluate_binary_classifier(
            y_valid,
            y_score,
            threshold=OPERATING_THRESHOLD,
        )
        diagnostic_curves = build_binary_diagnostic_curves(y_valid, y_score)
        threshold_frame = build_threshold_scenarios(y_valid, y_score, feature_set_name)
        threshold_frames.append(threshold_frame)

        validation_scores = bundle["valid_meta"].copy()
        validation_scores["feature_set"] = feature_set_name
        validation_scores["predicted_pd"] = y_score
        validation_scores["predicted_default_flag_at_050"] = (
            validation_scores["predicted_pd"] >= OPERATING_THRESHOLD
        ).astype(int)
        validation_scores["decision_at_050"] = np.where(
            validation_scores["predicted_pd"] >= OPERATING_THRESHOLD,
            "reject",
            "approve",
        )
        validation_scores_by_set[feature_set_name] = validation_scores

        consistency_checks = {
            "validation_score_rows_match_matrix": len(validation_scores) == bundle["X_valid"].shape[0],
            "validation_score_rows_match_meta": len(validation_scores) == len(bundle["valid_meta"]),
            "feature_name_count_matches_matrix": len(bundle["feature_names"]) == bundle["X_valid"].shape[1],
            "confusion_matrix_count_matches_meta": (
                int(np.array(primary_metrics["confusion_matrix"]).sum()) == len(bundle["valid_meta"])
            ),
        }
        operating_row = threshold_frame.loc[
            threshold_frame["threshold"] == OPERATING_THRESHOLD
        ].iloc[0].to_dict()
        metrics_payload_by_set[feature_set_name] = {
            "model_type": "logistic_regression_baseline",
            "feature_set": feature_set_name,
            "model_family": "logistic",
            "primary_metrics": primary_metrics,
            "business_summary": operating_row,
            "consistency_checks": consistency_checks,
            "pd_proxy_note": (
                "Predicted probabilities are uncalibrated PD proxies from the current logistic baseline."
            ),
        }
        artifacts_by_set[feature_set_name]["model"] = model
        artifacts_by_set[feature_set_name]["diagnostic_curves"] = diagnostic_curves
        artifacts_by_set[feature_set_name]["primary_metrics"] = primary_metrics

        comparison_rows.append(
            {
                "feature_set": feature_set_name,
                "model_family": "logistic",
                "model_label": f"logistic_{feature_set_name}",
                "feature_count": len(bundle["feature_names"]),
                "roc_auc": primary_metrics["roc_auc"],
                "average_precision": primary_metrics["average_precision"],
                "accuracy": primary_metrics["accuracy"],
                "precision": primary_metrics["precision"],
                "recall": primary_metrics["recall"],
                "f1": primary_metrics["f1"],
                "ks_statistic": diagnostic_curves["ks"]["statistic"],
                "ks_threshold": diagnostic_curves["ks"]["threshold"],
            }
        )

    comparison_metrics = pd.DataFrame(comparison_rows)
    threshold_scenarios = pd.concat(threshold_frames, ignore_index=True)
    operating_threshold_comparison = threshold_scenarios.loc[
        threshold_scenarios["threshold"] == OPERATING_THRESHOLD
    ].reset_index(drop=True)

    core_row = comparison_metrics.loc[
        comparison_metrics["feature_set"] == "traditional_core"
    ].iloc[0]
    proxy_row = comparison_metrics.loc[
        comparison_metrics["feature_set"] == "traditional_plus_proxy"
    ].iloc[0]
    core_operating_row = operating_threshold_comparison.loc[
        operating_threshold_comparison["feature_set"] == "traditional_core"
    ].iloc[0]
    proxy_operating_row = operating_threshold_comparison.loc[
        operating_threshold_comparison["feature_set"] == "traditional_plus_proxy"
    ].iloc[0]

    metric_columns = [
        "feature_count",
        "roc_auc",
        "average_precision",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "ks_statistic",
    ]
    threshold_metric_columns = [
        "approval_rate",
        "reject_rate",
        "approved_population_bad_rate",
        "rejected_population_bad_capture_rate",
    ]
    data_uplift_summary = pd.DataFrame(
        [
            {
                "comparison": "traditional_plus_proxy_minus_traditional_core",
                **{
                    f"{column}_delta": float(proxy_row[column] - core_row[column])
                    for column in metric_columns
                },
                **{
                    f"{column}_delta": float(proxy_operating_row[column] - core_operating_row[column])
                    for column in threshold_metric_columns
                },
            }
        ]
    )

    validation_score_comparison = artifacts_by_set[FEATURE_SET_NAMES[0]]["valid_meta"][
        [settings.id_column, settings.target_column]
    ].copy()
    for feature_set_name in FEATURE_SET_NAMES:
        validation_score_comparison = validation_score_comparison.merge(
            validation_scores_by_set[feature_set_name][
                [settings.id_column, "predicted_pd", "decision_at_050"]
            ].rename(
                columns={
                    "predicted_pd": f"{feature_set_name}_predicted_pd",
                    "decision_at_050": f"{feature_set_name}_decision_at_050",
                }
            ),
            on=settings.id_column,
            how="left",
        )

    display(comparison_metrics)
    display(data_uplift_summary)
    display(operating_threshold_comparison)


## 5. 可视化与落盘

这里至少输出三张对比图：ROC、PR、KS。所有结构化结果按 feature set 分目录保存，同时在根目录额外保存一份汇总 comparison 文件。


In [ ]:
if not PHASE_3_READY:
    print("图表与落盘已跳过，因为 Phase 2 产物不完整。")
elif not phase3_load_ok:
    print("图表与落盘已跳过，因为加载校验未通过。")
else:
    FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    MODELING_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    figure_paths = {
        "roc_comparison": FIGURE_OUTPUT_DIR / "phase3_logistic_feature_set_roc_comparison.png",
        "pr_comparison": FIGURE_OUTPUT_DIR / "phase3_logistic_feature_set_pr_comparison.png",
        "ks_comparison": FIGURE_OUTPUT_DIR / "phase3_logistic_feature_set_ks_comparison.png",
    }

    fig, ax = plt.subplots(figsize=(6, 5))
    for feature_set_name in FEATURE_SET_NAMES:
        bundle = artifacts_by_set[feature_set_name]
        metrics = bundle["primary_metrics"]
        curves = bundle["diagnostic_curves"]
        ax.plot(
            curves["roc"]["fpr"],
            curves["roc"]["tpr"],
            label=(
                f"logistic_{feature_set_name} ROC-AUC = {metrics['roc_auc']:.4f}"
            ),
        )
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
    ax.set_title("Phase 3 ROC Comparison by Feature Set")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(figure_paths["roc_comparison"], dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 5))
    for feature_set_name in FEATURE_SET_NAMES:
        bundle = artifacts_by_set[feature_set_name]
        metrics = bundle["primary_metrics"]
        curves = bundle["diagnostic_curves"]
        ax.plot(
            curves["precision_recall"]["recall"],
            curves["precision_recall"]["precision"],
            label=(
                f"logistic_{feature_set_name} AP = {metrics['average_precision']:.4f}"
            ),
        )
    ax.set_title("Phase 3 Precision-Recall Comparison by Feature Set")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(loc="lower left")
    fig.tight_layout()
    fig.savefig(figure_paths["pr_comparison"], dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 5))
    for feature_set_name in FEATURE_SET_NAMES:
        bundle = artifacts_by_set[feature_set_name]
        curves = bundle["diagnostic_curves"]
        ax.plot(
            curves["ks"]["thresholds"],
            curves["ks"]["values"],
            label=(
                f"logistic_{feature_set_name} KS = {curves['ks']['statistic']:.4f}"
            ),
        )
        ax.axvline(
            curves["ks"]["threshold"],
            linestyle="--",
            alpha=0.6,
        )
    ax.set_title("Phase 3 KS Comparison by Feature Set")
    ax.set_xlabel("Score Threshold")
    ax.set_ylabel("KS Statistic")
    ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(figure_paths["ks_comparison"], dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    for feature_set_name in FEATURE_SET_NAMES:
        output_dir = MODELING_OUTPUT_ROOT / feature_set_name
        output_dir.mkdir(parents=True, exist_ok=True)
        threshold_subset = threshold_scenarios.loc[
            threshold_scenarios["feature_set"] == feature_set_name
        ].reset_index(drop=True)
        validation_subset = validation_scores_by_set[feature_set_name]
        metrics_payload = {
            **metrics_payload_by_set[feature_set_name],
            "figure_paths": {key: str(path) for key, path in figure_paths.items()},
            "threshold_scenarios": threshold_subset.to_dict(orient="records"),
        }

        validation_subset.to_csv(output_dir / "validation_scores.csv", index=False)
        threshold_subset.to_csv(output_dir / "threshold_scenarios.csv", index=False)
        pd.DataFrame(
            {"feature_name": artifacts_by_set[feature_set_name]["feature_names"]}
        ).to_csv(output_dir / "feature_names.csv", index=False)
        joblib.dump(artifacts_by_set[feature_set_name]["model"], output_dir / "model.joblib")
        (output_dir / "metrics.json").write_text(
            json.dumps(_to_builtin(metrics_payload), indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

    comparison_metrics.to_csv(MODELING_OUTPUT_ROOT / "comparison_metrics.csv", index=False)
    data_uplift_summary.to_csv(MODELING_OUTPUT_ROOT / "data_uplift_summary.csv", index=False)
    threshold_scenarios.to_csv(MODELING_OUTPUT_ROOT / "threshold_scenarios.csv", index=False)
    operating_threshold_comparison.to_csv(
        MODELING_OUTPUT_ROOT / "operating_threshold_comparison.csv",
        index=False,
    )
    validation_score_comparison.to_csv(
        MODELING_OUTPUT_ROOT / "validation_score_comparison.csv",
        index=False,
    )
    summary_payload = {
        "comparison_type": "data_increment_with_fixed_logistic_model",
        "feature_sets": FEATURE_SET_NAMES,
        "operating_threshold": OPERATING_THRESHOLD,
        "comparison_metrics": comparison_metrics.to_dict(orient="records"),
        "data_uplift_summary": data_uplift_summary.to_dict(orient="records"),
        "figure_paths": {key: str(path) for key, path in figure_paths.items()},
    }
    (MODELING_OUTPUT_ROOT / "summary.json").write_text(
        json.dumps(_to_builtin(summary_payload), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


## 6. Checkpoint

到这里为止，Phase 3 已经把 logistic family 固定住，并把“是否加入 proxy”这个问题转化成可复现的数据增量对比。下一步进入 Phase 4 时，必须把这两套 data regime 继续保留下去，而不是重新退回单一数据口径。


In [ ]:
if not PHASE_3_READY:
    print("Checkpoint: Phase 3 仍被阻断，请先补齐两套 Phase 2 标准产物。")
elif not phase3_load_ok:
    print("Checkpoint: Phase 3 仍停在加载校验，请先修复 preprocessing artifact 的一致性问题。")
else:
    print("Phase 3 baseline comparison complete.")
    uplift_row = data_uplift_summary.iloc[0].to_dict()
    print(
        "业务提示：这里的 uplift 代表在同一 logistic 模型家族下，加入内部 proxy 变量后的数据增量，不代表模型家族升级。"
    )
    print(f"ROC-AUC delta: {uplift_row['roc_auc_delta']:.6f}")
    print(f"Average precision delta: {uplift_row['average_precision_delta']:.6f}")
    print("下一步如果进入 Phase 4，需要继续在同一 backend 下分别训练两套 feature set。")
